In [1]:
# ============================================================================
# 04_summary.ipynb
# Assemble a compact summary of the US replication results for the paper:
# descriptive statistics of the accounting variables and the headline
# regression / hedge-portfolio findings. Output saved to ../results/tables.
# ============================================================================

In [2]:
# --- Imports --------------------------------------------------------------
import pandas as pd
import numpy as np

DATA_DIR = "../data"
TAB_DIR  = "../results/tables"

In [3]:
# --- Descriptive statistics of accounting variables ----------------------
acc = pd.read_csv(f"{DATA_DIR}/accounting_panel.csv")

def winsorize(s, p=0.01):
    lo, hi = s.quantile(p), s.quantile(1 - p)
    return s.clip(lo, hi)

desc_vars = ["tacc", "gpoa"]
desc = pd.DataFrame({v: winsorize(acc[v].dropna()) for v in desc_vars}).describe(
    percentiles=[0.25, 0.5, 0.75]).T.round(4)
desc.to_csv(f"{TAB_DIR}/descriptive_statistics.csv")
desc

,count,mean,std,min,25%,50%,75%,max
tacc,20893.0,-1.6257,9.8049,-85.6155,-0.1266,-0.0513,-0.0084,1.4047
gpoa,3591.0,0.6730,1.9538,-0.4125,0.1282,0.2920,0.5447,17.3152


In [4]:
# --- Headline results (assembled from notebook 03 outputs) ---------------
reg = pd.read_csv(f"{TAB_DIR}/reg_gpoa_on_tacc.csv", index_col=0)
alphas = pd.read_csv(f"{TAB_DIR}/tacc_hedge_alphas.csv")

print("Predictive regression: next-year GPOA on current TACC")
print(reg.to_string())
print("\nLow-minus-high TACC hedge alphas:")
print(alphas.to_string(index=False))

Predictive regression: next-year GPOA on current TACC
         coef        t       p
const  0.3051  15.6952  0.0000
tacc   0.0080   1.1390  0.2547
gpoa   0.2969   7.5441  0.0000

Low-minus-high TACC hedge alphas:
   model  alpha(%)  t(alpha)
    CAPM     2.293      2.87
     FF3     2.281      2.77
Carhart4     2.389      3.08


In [5]:
# --- Quintile monotonicity check -----------------------------------------
w = pd.read_csv(f"{TAB_DIR}/tacc_portfolio_returns.csv", index_col=0, parse_dates=True)
q_means = (w[[f"P{i}" for i in range(1, 6)]].mean() * 100).round(3)
q_means.name = "mean_monthly_return_pct"
q_means.to_csv(f"{TAB_DIR}/quintile_mean_returns.csv")
print("Mean monthly return by quintile (P1=high TACC ... P5=low TACC):")
print(q_means.to_string())

Mean monthly return by quintile (P1=high TACC ... P5=low TACC):
P1    1.688
P2    1.212
P3    1.399
P4    1.720
P5    3.949
